In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

dossier_projet = (
    Path.home()
    / "Documents"
    / "jedha_exercices"
    / "prospection-b2b"
)

collecte_id = "20260917T133858808210Z"
dossier_collecte = dossier_projet / "data" / "raw" / "sirene" / collecte_id

if not dossier_collecte.is_dir():
    raise FileNotFoundError(f"Dossier introuvable : {dossier_collecte}")

fichiers_json = sorted(dossier_collecte.glob("page_*.json"))

print("Python utilisé :", sys.executable)
print("Version DuckDB :", duckdb.__version__)
print("Pages JSON trouvées :", len(fichiers_json))

# Une base temporaire pour explorer les données.
connexion = duckdb.connect(":memory:")

connexion.sql("SELECT 'Prêt pour l’exploration' AS statut").df()

In [ ]:
import json

# Chaque page contient une liste "etablissements".
etablissements = []

for fichier in fichiers_json:
    contenu = json.loads(fichier.read_text(encoding="utf-8"))
    etablissements.extend(contenu["etablissements"])

# Préparer les champs utiles à notre exploration.
lignes = []

date_reference = "2026-09-17"

for etablissement in etablissements:
    unite_legale = etablissement.get("uniteLegale") or {}
    adresse = etablissement.get("adresseEtablissement") or {}

    periodes_courantes = [
        periode
        for periode in (etablissement.get("periodesEtablissement") or [])
        if periode.get("dateDebut") is not None
        and periode["dateDebut"] <= date_reference
        and (
            periode.get("dateFin") is None
            or periode["dateFin"] >= date_reference
        )
    ]

    periode_courante = (
        periodes_courantes[0] if len(periodes_courantes) == 1 else {}
    )

    lignes.append({
        "siret": etablissement.get("siret"),
        "siren": etablissement.get("siren"),
        "siege": etablissement.get("etablissementSiege"),
        "date_creation_entreprise": unite_legale.get("dateCreationUniteLegale"),
        "date_creation_etablissement": etablissement.get("dateCreationEtablissement"),
        "naf_etablissement": periode_courante.get("activitePrincipaleEtablissement"),
        "statut_etablissement": periode_courante.get("etatAdministratifEtablissement"),
        "statut_entreprise": unite_legale.get("etatAdministratifUniteLegale"),
        "diffusion_etablissement": etablissement.get("statutDiffusionEtablissement"),
        "diffusion_entreprise": unite_legale.get("statutDiffusionUniteLegale"),
        "code_commune": adresse.get("codeCommuneEtablissement"),
        "code_postal": adresse.get("codePostalEtablissement"),
        "tranche_effectifs": etablissement.get("trancheEffectifsEtablissement"),
        "annee_effectifs": etablissement.get("anneeEffectifsEtablissement"),
        "nombre_periodes_courantes": len(periodes_courantes),
    })

df_etablissements = pd.DataFrame(lignes)

# Conserver les identifiants et codes comme du texte.
colonnes_texte = [
    colonne
    for colonne in df_etablissements.columns
    if colonne not in ["siege", "nombre_periodes_courantes"]
]

df_etablissements[colonnes_texte] = (
    df_etablissements[colonnes_texte].astype("string")
)

# Rendre le tableau accessible au SQL dans DuckDB.
connexion.register("etablissements_bruts", df_etablissements)

print(f"Établissements chargés : {len(df_etablissements):,}")

In [ ]:
connexion.sql("""
    SELECT
        COUNT(*) AS nombre_lignes,
        COUNT(DISTINCT siret) AS sirets_distincts,
        COUNT(DISTINCT siren) AS sirens_distincts,
        COUNT(*) FILTER (
            WHERE siret IS NULL OR TRIM(siret) = ''
        ) AS sirets_manquants,
        COUNT(siret) - COUNT(DISTINCT siret) AS repetitions_siret,
        COUNT(*) FILTER (
            WHERE nombre_periodes_courantes <> 1
        ) AS anomalies_periode_courante
    FROM etablissements_bruts
""").df()

In [ ]:
champs_a_examiner = [
    "date_creation_entreprise",
    "naf_etablissement",
    "statut_etablissement",
    "statut_entreprise",
    "diffusion_etablissement",
    "diffusion_entreprise",
    "code_commune",
    "code_postal",
    "tranche_effectifs",
    "annee_effectifs",
]

profil = []

for champ in champs_a_examiner:
    valeurs = df_etablissements[champ].str.strip()

    absentes = valeurs.isna() | valeurs.eq("").fillna(False)
    masquees = valeurs.eq("[ND]").fillna(False)

    profil.append({
        "champ": champ,
        "valeurs_absentes": int(absentes.sum()),
        "valeurs_masquees": int(masquees.sum()),
        "pourcentage_indisponible": round(
            100 * (absentes | masquees).mean(), 1
        ),
    })

df_qualite = (
    pd.DataFrame(profil)
    .sort_values("pourcentage_indisponible", ascending=False)
    .reset_index(drop=True)
)

display(df_qualite)

In [ ]:
graphique = df_qualite.set_index("champ")[
    ["valeurs_absentes", "valeurs_masquees"]
].copy()

graphique = graphique / len(df_etablissements) * 100

ax = graphique.plot.barh(
    stacked=True,
    figsize=(10, 6),
    color=["#64748B", "#F59E0B"],
)

ax.invert_yaxis()
ax.set_title("Disponibilité des champs — 10 607 établissements")
ax.set_xlabel("Part des établissements (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.legend(["Valeur absente", "Valeur masquée [ND]"])
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
connexion.sql("""
    SELECT
        tranche_effectifs,
        annee_effectifs,
        COUNT(*) AS nombre_etablissements,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            1
        ) AS pourcentage
    FROM etablissements_bruts
    GROUP BY tranche_effectifs, annee_effectifs
    ORDER BY nombre_etablissements DESC
""").df()

### Effectifs : limite constatée et décision

Les 10 607 établissements présentent la même tranche d’effectifs :
`NN`. L’année associée est absente pour tous.

Ce champ ne permet donc pas de différencier les prospects.

Décisions :
- Conserver la valeur brute pour la traçabilité.
- Ne pas convertir `NN` en un effectif numérique.
- Ne pas filtrer les entreprises sur ce critère.
- Exclure les effectifs du score dans cette première version.

In [ ]:
connexion.sql("""
    SELECT
        diffusion_entreprise,
        diffusion_etablissement,
        COUNT(*) AS nombre_etablissements,
        COUNT(*) FILTER (
            WHERE TRIM(code_postal) = '[ND]'
        ) AS codes_postaux_masques
    FROM etablissements_bruts
    GROUP BY diffusion_entreprise, diffusion_etablissement
    ORDER BY nombre_etablissements DESC
""").df()

### Diffusion : séparation des périmètres

- Collecte brute : 10 607 établissements.
- Sélection pour qualification commerciale : 7 370 établissements
  dont les deux statuts de diffusion sont `O`.
- Exclus de cette sélection : 3 237 établissements présentant
  au moins un statut `P`.

Les codes postaux masqués ne seront pas reconstitués.
Les statuts de diffusion seront conservés dans les transformations.

In [ ]:
# Lire tous les codes comme du texte pour préserver les zéros initiaux.
df_communes = pd.read_csv(
    dossier_projet / "data" / "raw" / "v_commune_2026.csv",
    dtype="string",
)

# Conserver les communes et les arrondissements municipaux,
# notamment ceux de Paris.
df_communes_jointure = df_communes.loc[
    df_communes["TYPECOM"].isin(["COM", "ARM"]),
    ["COM", "TYPECOM", "LIBELLE", "DEP", "REG", "COMPARENT"],
].copy()

if df_communes_jointure["COM"].isna().any():
    raise ValueError("Le référentiel contient un code commune absent.")

if df_communes_jointure["COM"].duplicated().any():
    raise ValueError("Plusieurs lignes correspondent au même code commune.")

connexion.register("referentiel_communes", df_communes_jointure)

print("Référentiel prêt : les codes commune sont renseignés et uniques.")

In [ ]:
connexion.sql("""
    SELECT
        COUNT(*) AS lignes_apres_jointure,
        COUNT(DISTINCT etablissements_bruts.siret) AS sirets_distincts,
        COUNT(*) FILTER (
            WHERE referentiel_communes.COM IS NOT NULL
        ) AS etablissements_avec_correspondance,
        COUNT(*) FILTER (
            WHERE referentiel_communes.COM IS NULL
        ) AS etablissements_sans_correspondance,
        COUNT(*) FILTER (
            WHERE referentiel_communes.REG <> '11'
        ) AS etablissements_hors_ile_de_france
    FROM etablissements_bruts
    LEFT JOIN referentiel_communes
        ON etablissements_bruts.code_commune = referentiel_communes.COM
""").df()

### Enrichissement géographique

- 10 607 établissements avant et après jointure.
- 10 607 SIRET distincts après jointure.
- Tous les établissements trouvent une correspondance dans le COG 2026.
- Aucun établissement perdu ni dupliqué par la jointure.

Décisions pour dbt :
- Conserver une jointure gauche sur le code commune.
- Tester l’unicité des codes dans le référentiel.
- Tester les correspondances et la conservation du nombre de lignes.
- Conserver les codes géographiques au format texte.

### Codes postaux masqués et enrichissement géographique

Les 3 237 codes postaux masqués ne provoquent aucune exclusion
de la base enrichie. Grâce au code commune disponible, la jointure avec le COG 2026
permet de rattacher les 10 607 établissements à leur commune,
leur département et leur région, sans reconstituer les codes
postaux masqués.


In [ ]:
df_activites = connexion.sql("""
    SELECT
        naf_etablissement,
        COUNT(*) AS nombre_etablissements
    FROM etablissements_bruts
    GROUP BY naf_etablissement
    ORDER BY nombre_etablissements DESC
""").df()

display(df_activites)

ax = df_activites.set_index("naf_etablissement").plot.bar(
    y="nombre_etablissements",
    figsize=(9, 5),
    color="#2563EB",
    legend=False,
    rot=0,
)

ax.set_title("Répartition des 10 607 établissements par activité")
ax.set_xlabel("Code NAF à la date du 17 septembre 2026")
ax.set_ylabel("Nombre d’établissements")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.0f", padding=3)

ax.margins(y=0.15)
plt.tight_layout()
plt.show()

### Sélection des périodes à la date de référence

Un établissement présente un changement d’activité prenant effet
le 1er octobre 2026, après notre date de référence du 17 septembre 2026.

Sélectionner uniquement la période sans date de fin faisait apparaître
cette activité future.

Correction : sélectionner la période dont les dates de début et de fin
encadrent la date de référence, une date de fin absente étant admise.

Après correction : 10 607 établissements dans les trois activités ciblées.

À reproduire dans dbt : sélection temporelle et contrôle de l’existence
d’une seule période applicable par établissement.

In [ ]:
# idenfitier si on n'a pas raté nos établissements cibles par les NAF

naf_cibles = ["58.29C", "62.02A", "62.01Z"]

hors_cible = df_etablissements.loc[
    ~df_etablissements["naf_etablissement"].isin(naf_cibles)
]

display(
    hors_cible[
        ["siret", "date_creation_entreprise", "naf_etablissement"]
    ]
)

# Examiner les périodes présentes dans les réponses brutes.
sirets_hors_cible = set(hors_cible["siret"])
historique = []

for etablissement in etablissements:
    if etablissement.get("siret") in sirets_hors_cible:
        for periode in etablissement.get("periodesEtablissement") or []:
            historique.append({
                "siret": etablissement.get("siret"),
                "date_debut": periode.get("dateDebut"),
                "date_fin": periode.get("dateFin"),
                "naf": periode.get("activitePrincipaleEtablissement"),
                "statut": periode.get("etatAdministratifEtablissement"),
            })

display(pd.DataFrame(historique))

In [ ]:
# Vérifier la présence d'anomalies dans les donnée collectées 

connexion.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE nombre_periodes_courantes <> 1
        ) AS anomalies_periode,

        COUNT(*) FILTER (
            WHERE TRY_CAST(date_creation_entreprise AS DATE) IS NULL
               OR TRY_CAST(date_creation_entreprise AS DATE)
                  NOT BETWEEN DATE '2026-03-17' AND DATE '2026-09-17'
        ) AS anomalies_date_creation,

        COUNT(*) FILTER (
            WHERE siege IS DISTINCT FROM TRUE
        ) AS anomalies_siege,

        COUNT(*) FILTER (
            WHERE statut_etablissement IS DISTINCT FROM 'A'
        ) AS anomalies_statut_etablissement,

        COUNT(*) FILTER (
            WHERE statut_entreprise IS DISTINCT FROM 'A'
        ) AS anomalies_statut_entreprise,

        COUNT(*) FILTER (
            WHERE referentiel_communes.REG IS DISTINCT FROM '11'
        ) AS anomalies_region

    FROM etablissements_bruts
    LEFT JOIN referentiel_communes
        ON etablissements_bruts.code_commune = referentiel_communes.COM
""").df().T.rename(columns={0: "nombre_anomalies"})

In [ ]:
# Créations mensuelles

df_creations = connexion.sql("""
    SELECT
        STRFTIME(
            CAST(date_creation_entreprise AS DATE),
            '%Y-%m'
        ) AS mois_creation,
        naf_etablissement,
        COUNT(*) AS nombre_etablissements
    FROM etablissements_bruts
    GROUP BY mois_creation, naf_etablissement
    ORDER BY mois_creation, naf_etablissement
""").df()

# Une ligne par mois et une colonne par activité.
df_creations_par_naf = (
    df_creations.pivot(
        index="mois_creation",
        columns="naf_etablissement",
        values="nombre_etablissements",
    )
    .reindex(columns=["58.29C", "62.01Z", "62.02A"])
    .fillna(0)
    .astype(int)
)

display(df_creations_par_naf)

ax = df_creations_par_naf.plot.bar(
    stacked=True,
    figsize=(11, 6),
    color=["#14B8A6", "#2563EB", "#F59E0B"],
    rot=0,
)

ax.set_title("Créations par mois et par activité — 10 607 établissements")
ax.set_xlabel(
    "Mois de création — mars : du 17 au 31 ; septembre : du 1er au 17"
)
ax.set_ylabel("Nombre d’établissements")
ax.legend(
    title="Activité NAF",
    labels=[
        "58.29C — Édition de logiciels applicatifs",
        "62.01Z — Programmation informatique",
        "62.02A — Conseil en informatique",
    ],
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    fontsize=9,
)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

# Afficher le total au-dessus de chaque colonne.
totaux = df_creations_par_naf.sum(axis=1)

for position, total in enumerate(totaux):
    ax.annotate(
        str(total),
        xy=(position, total),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
    )

ax.margins(y=0.15)
plt.tight_layout()
plt.show()

### Collecte complète et candidats à qualifier

- **Collecte complète — 10 607 entreprises** : ensemble des entreprises
  récupérées via l’API Sirene, correspondant à notre périmètre :
  créations du 17 mars au 17 septembre 2026, sièges en Île-de-France,
  trois activités NAF ciblées et statuts actifs à la date de référence.


In [ ]:
bilans = []

for chemin in sorted(
    (dossier_projet / "data" / "raw" / "sirene")
    .glob("2026-*/*/collecte.json")
):
    metadata = json.loads(chemin.read_text(encoding="utf-8"))

    bilans.append({
        "mois": metadata.get("mois_collecte"),
        "collecte_id": metadata.get("collecte_id"),
        "statut": metadata.get("statut"),
        "etablissements": metadata.get("total_recu"),
        "pages": metadata.get("nombre_pages"),
    })

df_collectes = pd.DataFrame(bilans)
display(df_collectes)

In [ ]:
# Retenir la dernière collecte terminée de chaque mois.
df_collectes_retenues = (
    df_collectes.loc[df_collectes["statut"].eq("terminee")]
    .sort_values(["mois", "collecte_id"])
    .drop_duplicates(subset="mois", keep="last")
    .sort_values("mois")
    .reset_index(drop=True)
)

display(df_collectes_retenues)

print("Mois retenus :", len(df_collectes_retenues))
print(
    "Total des lignes annoncées :",
    int(df_collectes_retenues["etablissements"].sum()),
)
print(
    "Total des pages :",
    int(df_collectes_retenues["pages"].sum()),
)

In [ ]:
#Chargements des lignes par mois

lignes_mensuelles = []

for collecte in df_collectes_retenues.to_dict("records"):
    dossier = (
        dossier_projet
        / "data"
        / "raw"
        / "sirene"
        / collecte["mois"]
        / collecte["collecte_id"]
    )

    metadata = json.loads(
        (dossier / "collecte.json").read_text(encoding="utf-8")
    )
    date_reference_mois = metadata["date_reference"]
    nombre_lignes_mois = 0

    for numero_page in range(1, int(metadata["nombre_pages"]) + 1):
        fichier = dossier / f"page_{numero_page:04d}.json"
        contenu = json.loads(fichier.read_text(encoding="utf-8"))

        for etablissement in contenu["etablissements"]:
            unite_legale = etablissement.get("uniteLegale") or {}
            adresse = etablissement.get("adresseEtablissement") or {}

            periodes_applicables = [
                periode
                for periode in (
                    etablissement.get("periodesEtablissement") or []
                )
                if periode.get("dateDebut") is not None
                and periode["dateDebut"] <= date_reference_mois
                and (
                    periode.get("dateFin") is None
                    or periode["dateFin"] >= date_reference_mois
                )
            ]

            periode = (
                periodes_applicables[0]
                if len(periodes_applicables) == 1
                else {}
            )

            lignes_mensuelles.append({
                "mois_collecte": collecte["mois"],
                "collecte_id": collecte["collecte_id"],
                "date_reference": date_reference_mois,
                "siret": etablissement.get("siret"),
                "siren": etablissement.get("siren"),
                "siege": etablissement.get("etablissementSiege"),
                "date_creation_entreprise": unite_legale.get(
                    "dateCreationUniteLegale"
                ),
                "naf_etablissement": periode.get(
                    "activitePrincipaleEtablissement"
                ),
                "statut_etablissement": periode.get(
                    "etatAdministratifEtablissement"
                ),
                "statut_entreprise": unite_legale.get(
                    "etatAdministratifUniteLegale"
                ),
                "diffusion_etablissement": etablissement.get(
                    "statutDiffusionEtablissement"
                ),
                "diffusion_entreprise": unite_legale.get(
                    "statutDiffusionUniteLegale"
                ),
                "code_commune": adresse.get("codeCommuneEtablissement"),
                "code_postal": adresse.get("codePostalEtablissement"),
                "tranche_effectifs": etablissement.get(
                    "trancheEffectifsEtablissement"
                ),
                "annee_effectifs": etablissement.get(
                    "anneeEffectifsEtablissement"
                ),
                "nombre_periodes_applicables": len(periodes_applicables),
            })

            nombre_lignes_mois += 1

    if nombre_lignes_mois != int(metadata["total_recu"]):
        raise ValueError(
            f"Écart de volume pour {collecte['mois']} : "
            f"{nombre_lignes_mois} lignes lues, "
            f"{metadata['total_recu']} attendues."
        )

df_etablissements_mensuels = pd.DataFrame(lignes_mensuelles)

colonnes_texte = [
    colonne
    for colonne in df_etablissements_mensuels.columns
    if colonne not in ["siege", "nombre_periodes_applicables"]
]

df_etablissements_mensuels[colonnes_texte] = (
    df_etablissements_mensuels[colonnes_texte].astype("string")
)

connexion.register(
    "etablissements_mensuels",
    df_etablissements_mensuels,
)

print("Lignes chargées :", len(df_etablissements_mensuels))

In [ ]:
df_creations = connexion.sql("""
    SELECT
        mois_collecte,
        naf_etablissement,
        COUNT(*) AS nombre_etablissements
    FROM etablissements_mensuels
    GROUP BY mois_collecte, naf_etablissement
    ORDER BY mois_collecte, naf_etablissement
""").df()

df_creations_par_naf = (
    df_creations.pivot(
        index="mois_collecte",
        columns="naf_etablissement",
        values="nombre_etablissements",
    )
    .fillna(0)
    .astype(int)
    .sort_index()
)

# Vérifier les activités avant de sélectionner les colonnes du graphique.
naf_cibles = ["58.29C", "62.01Z", "62.02A"]

if (
    df_creations["naf_etablissement"].isna().any()
    or not df_creations["naf_etablissement"].isin(naf_cibles).all()
):
    raise ValueError(
        "Une activité est absente ou hors cible : examiner df_creations."
    )

df_creations_par_naf = df_creations_par_naf.reindex(
    columns=naf_cibles,
    fill_value=0,
)

# Afficher les volumes mensuels et leur cumul.
df_bilan_mensuel = df_creations_par_naf.copy()
df_bilan_mensuel["total_mois"] = df_creations_par_naf.sum(axis=1)
df_bilan_mensuel["cumul"] = df_bilan_mensuel["total_mois"].cumsum()

display(df_bilan_mensuel)

total = int(df_bilan_mensuel["total_mois"].sum())

ax = df_creations_par_naf.plot.bar(
    stacked=True,
    figsize=(11, 6),
    color=["#14B8A6", "#2563EB", "#F59E0B"],
    rot=0,
)

ax.set_title(
    f"Créations par mois et par activité — {total:,} établissements"
    .replace(",", " ")
)
ax.set_xlabel("Mois de création — mois complets")
ax.set_ylabel("Nombre d’établissements")
ax.legend(
    title="Code NAF à la fin du mois de création",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

for position, total_mois in enumerate(df_bilan_mensuel["total_mois"]):
    ax.annotate(
        str(total_mois),
        xy=(position, total_mois),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
    )

ax.margins(y=0.15)
plt.tight_layout()
plt.show()

In [ ]:
df_controles_mensuels = connexion.sql("""
    SELECT
        COUNT(*) AS lignes_apres_jointure,
        COUNT(DISTINCT etablissements_mensuels.siret) AS sirets_distincts,

        COUNT(*) FILTER (
            WHERE etablissements_mensuels.siret IS NULL
               OR TRIM(etablissements_mensuels.siret) = ''
        ) AS sirets_manquants,

        COUNT(*) FILTER (
            WHERE nombre_periodes_applicables <> 1
        ) AS anomalies_periode,

        COUNT(*) FILTER (
            WHERE TRY_CAST(date_creation_entreprise AS DATE) IS NULL
               OR STRFTIME(
                    TRY_CAST(date_creation_entreprise AS DATE), '%Y-%m'
                  ) IS DISTINCT FROM mois_collecte
        ) AS anomalies_mois_creation,

        COUNT(*) FILTER (
            WHERE TRY_CAST(date_reference AS DATE) IS NULL
               OR TRY_CAST(date_reference AS DATE)
                  IS DISTINCT FROM LAST_DAY(
                      TRY_CAST(mois_collecte || '-01' AS DATE)
                  )
        ) AS anomalies_date_reference,

        COUNT(*) FILTER (
            WHERE siege IS DISTINCT FROM TRUE
        ) AS anomalies_siege,

        COUNT(*) FILTER (
            WHERE statut_etablissement IS DISTINCT FROM 'A'
        ) AS anomalies_statut_etablissement,

        COUNT(*) FILTER (
            WHERE statut_entreprise IS DISTINCT FROM 'A'
        ) AS anomalies_statut_entreprise,

        COUNT(*) FILTER (
            WHERE naf_etablissement IS NULL
               OR naf_etablissement NOT IN ('58.29C', '62.01Z', '62.02A')
        ) AS anomalies_naf,

        COUNT(*) FILTER (
            WHERE referentiel_communes.COM IS NULL
        ) AS communes_sans_correspondance,

        COUNT(*) FILTER (
            WHERE referentiel_communes.REG IS DISTINCT FROM '11'
        ) AS region_absente_ou_hors_idf,

        COUNT(*) FILTER (
            WHERE code_postal IS NOT NULL
              AND TRIM(code_postal) NOT IN ('', '[ND]')
              AND NOT REGEXP_FULL_MATCH(TRIM(code_postal), '[0-9]{5}')
        ) AS anomalies_format_code_postal

    FROM etablissements_mensuels
    LEFT JOIN referentiel_communes
        ON etablissements_mensuels.code_commune = referentiel_communes.COM
""").df()

display(
    df_controles_mensuels.T.rename(columns={0: "resultat"})
)

In [ ]:
df_effectifs_mensuels = connexion.sql("""
    SELECT
        tranche_effectifs,
        annee_effectifs,
        COUNT(*) AS nombre_etablissements,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            1
        ) AS pourcentage
    FROM etablissements_mensuels
    GROUP BY tranche_effectifs, annee_effectifs
    ORDER BY nombre_etablissements DESC
""").df()

display(df_effectifs_mensuels)

### Bilan de l’exploration — janvier à août 2026

L’historique contient 14 578 établissements distincts, issus de huit collectes mensuelles. Seule la dernière exécution terminée de chaque mois est retenue.

Les contrôles réalisés ne détectent aucune anomalie sur les identifiants, les périodes applicables, les dates de création, les sièges, les statuts, les activités ciblées et le rattachement à l’Île-de-France.

| Constat | Transformation prévue | Contrôle à automatiser |
|---|---|---|
| Plusieurs exécutions possibles pour un mois | Retenir la dernière collecte terminée | Une collecte retenue par mois |
| SIRET renseignés et uniques | Conserver les identifiants comme du texte | SIRET non nul, unique et composé de 14 chiffres |
| Présence possible de périodes futures | Sélectionner la période applicable à la date de référence du mois | Exactement une période applicable par établissement |
| Dates de création conformes au mois collecté | Convertir les dates en type DATE | Date valide et comprise dans le mois |
| Trois activités NAF ciblées | Conserver le NAF applicable à la date de référence | Valeur parmi 58.29C, 62.01Z et 62.02A |
| Sièges et statuts actifs conformes | Conserver ces critères dans la sélection | Siège vrai et statuts égaux à A |
| Tous les codes commune trouvent une correspondance | Enrichir par jointure gauche avec le COG | Correspondance, région 11 et conservation du nombre de lignes |
| Des codes postaux peuvent être masqués | Convertir [ND] en NULL dans le champ nettoyé et conserver un indicateur de masquage | Code disponible composé de cinq chiffres |
| Effectifs tous égaux à NN, année absente | Conserver la valeur brute ; exclure ce critère du score | Suivre sa disponibilité lors des prochaines collectes |
| Statuts de diffusion disponibles | Les conserver sans exclusion de la première sélection | Suivre les valeurs et leur répartition |

La jointure avec le référentiel COG utilise le code commune INSEE pour retrouver le nom de la commune, le département et la région.
Le COG utilisé ne contient pas de codes postaux : les codes postaux masqués dans Sirene restent donc indisponibles.
Cela n’empêche pas la sélection géographique, car le rattachement à l’Île-de-France est vérifié grâce au code région.

L’historique décrit les critères évalués à la fin de chaque mois de création pour les champs historisés. Il ne garantit pas que toutes les entreprises sont encore actives aujourd’hui.

Le score prévu sera un score de priorité fondé sur des règles métier, pas une probabilité de conversion.

In [ ]:
from pathlib import Path
import duckdb

chemin_base = (
    Path.home()
    / "Documents/jedha_exercices/prospection-b2b"
    / "data/warehouse/prospection.duckdb"
)

with duckdb.connect(str(chemin_base), read_only=True) as connexion_scores:
    df_repartition_scores = connexion_scores.sql("""
        select
            couleur_priorite,
            segment_anciennete,
            naf_etablissement,
            score_total,
            count(*) as nombre_etablissements

        from analytics.mart_prospects_scores

        group by
            couleur_priorite,
            segment_anciennete,
            naf_etablissement,
            score_total

        order by
            score_total desc nulls last,
            segment_anciennete,
            naf_etablissement
    """).df()

display(df_repartition_scores)

print(
    "Total des établissements :",
    df_repartition_scores["nombre_etablissements"].sum()
)

In [ ]:
from pathlib import Path
import duckdb

chemin_base = (
    Path.home()
    / "Documents/jedha_exercices/prospection-b2b"
    / "data/warehouse/prospection.duckdb"
)

with duckdb.connect(str(chemin_base), read_only=True) as connexion_scores:
    df_repartition_age = connexion_scores.sql("""
        select
            libelle_age,
            coalesce(couleur_priorite, 'Hors périmètre V1') as priorite,
            count(*) as nombre_etablissements,
            min(score_total) as score_minimum,
            max(score_total) as score_maximum

        from analytics.mart_prospects_scores

        group by
            libelle_age,
            couleur_priorite

        order by min(anciennete_jours)
    """).df()

display(df_repartition_age)

print(
    "Total des établissements :",
    df_repartition_age["nombre_etablissements"].sum()
)